In [1]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import numpy.random as nr
import jax
import scipy
from gestimators import BF
from IMH2 import IndMH
import matplotlib.pyplot as plt
from gdistribution import gprior, mix_normal

In [8]:
# def load_data():
#         file = '/home/ranbot/optionpricing/data/ripley.mat'
#         mat = scipy.io.loadmat(file)
#         data = mat['X']
#         inputs = data[:, 0:-1]
#         print(inputs.shape)
#         outputs = data[:, -1]
#         return inputs, outputs

# input, output = load_data()

In [2]:
n_predata = 50
def gen_data():
    X = nr.multivariate_normal(np.zeros(5), np.eye(5), n_predata)
    y = np.zeros(n_predata)
    for i in range(n_predata):
        y[i] = nr.normal(5*X[i,3] + 10*X[i,4], 2.5)
    return X, y

input, output = gen_data()
input = input[:, 3:]
dim = 2

In [3]:
cov0 = np.linalg.inv(input.T @ input)
print(cov0)

[[ 0.02454044 -0.00394637]
 [-0.00394637  0.0166342 ]]


In [4]:
nc = 10
tar = gprior(dim)
tar.set_parameters(cov0)
pro = mix_normal(dim)
a = tar.sampler(100000)

pro.em_algorithm(a,nc,1000)
weights = pro.weights
covs = pro.covs

# covs = np.array([np.cov(a[:,0], a[:,1])])
b = pro.sampler(10000)
plt.hist(a[:,0], bins = 50, density = True)
plt.show()
plt.hist(b[:,0], bins = 50, density = True)
plt.show()

I0000 00:00:1701166129.518075   11775 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.
/home/ranbot/optionpricing/gexp/gdistribution.py:76: RuntimeWarning: invalid value encountered in divide
  loglik = loglik / np.sum(loglik, axis = 1)[:, np.newaxis]


KeyboardInterrupt: 

In [12]:
Itrnum = 2000
sample_size = 1000
target = 'gprior'
proposal = 'mix_normal'

In [14]:
epoch = 5
cmc_mean = []
imhcv_mean = []
MyIM = IndMH(Itrnum, dim, target, proposal)
MyIM.set_distribution_parameters(cov0, nc, weights, covs)
est = BF(sample_size, weights, covs, input, output)
estcmc = BF(sample_size, weights, covs, input, output)
tar = gprior(dim)
print("start")
for i in range(epoch):
    data = MyIM.IMH()
    print(data[2].mean())
    est.sample_loader(data[0],data[1],data[2])
    m,v= est.IMCV()
    imhcv_mean.append(m)
    datacmc = tar.sampler(cov0, sample_size)
    estcmc.cmc_loader(datacmc)
    mc,vc = estcmc.CMC()
    cmc_mean.append(mc)
    if (i + 1) % 1 == 0:
        print(i + 1, "replications completed !")

start
0.88980525561564
[[ 4.10061419e-123 -3.58363133e-123]
 [-3.58363133e-123  7.23243691e-123]]
[[ 7.23243691e-123 -3.43705587e-108]
 [-3.43705587e-108  1.59468660e-058]]
[[ 1.         -0.65804644]
 [-0.65804644  1.        ]]
[[ 1.00000000e+00 -3.20041718e-18]
 [-3.20041718e-18  1.00000000e+00]]
0.5577303248215256 -2.157474937155502e-50
1 replications completed !
0.8916851394449478
[[ 3.20886919e-155 -2.08193307e-155]
 [-2.08193307e-155  4.55966777e-155]]
[[4.55966777e-155 1.99880401e-124]
 [1.99880401e-124 1.59468660e-058]]
[[ 1.         -0.54428223]
 [-0.54428223  1.        ]]
[[1.00000000e+00 2.34404497e-18]
 [2.34404497e-18 1.00000000e+00]]
0.6350945626495896 1.2546696120523457e-66
2 replications completed !
0.8920474630109966
[[ 3.43240007e-110 -3.06617586e-110]
 [-3.06617586e-110  6.17142303e-110]]
[[ 6.17142303e-110 -2.76879829e-101]
 [-2.76879829e-101  1.59468660e-058]]
[[ 1.         -0.66620126]
 [-0.66620126  1.        ]]
[[ 1.00000000e+00 -8.82594033e-18]
 [-8.82594033e-18

In [ ]:
print(np.array(imhcv_mean).var())
print(np.array(cmc_mean).var())
print(np.array(imhcv_mean).mean())
print(np.array(cmc_mean).mean())
print(np.array(cmc_mean).var()/np.array(imhcv_mean).var())

2.256463009364131e-23
1.6204657577860837e-22
1.865042512259235e-12
5.562001376104894e-11
7.181441712367044
